# Solar filament instance segmentation: end-to-end pipeline

Test images -> Mask R-CNN detector (translate TTA, mask NMS, score threshold, pixel ownership)
-> seed-box crop U-Net refiner -> disjoint instance masks -> submission CSV, then every quantity of
the committee rubric on the 60-image validation split. All steps call `filseg/`; the same code runs
from the command line as `scripts/predict.py` and `scripts/rubric_metrics.py`.

Needs the challenge data under `data/` and the two checkpoints under `weights/`:
`kaggle competitions download -c filament-segmentation-2026 -p data --unzip`, then
`kaggle datasets download -d mainguyenanhvu/filament-seg-2026-weights -p weights --unzip`
and `sha256sum -c weights/SHA256SUMS` (see README).

In [ ]:
import sys, time
from pathlib import Path

import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from filseg.infer import Config, encode, predict_all, submission_rows
from filseg.io import heldout_stems, load_records
from filseg.metrics import evaluate
from filseg.model import load_detector, load_refiner

DATA = ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026"
TEST_IMAGES = DATA / "test" / "test_images"
TRAIN_IMAGES = DATA / "train" / "train_images"
ANN = DATA / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
DETECTOR, REFINER = ROOT / "weights" / "detector.pt", ROOT / "weights" / "refiner.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config()  # score 0.80, refine 0.40, min area 250 px, NMS IoU 0.1, TTA shift 16 px
cfg, device

In [ ]:
detector = load_detector(DETECTOR, device)
refiner = load_refiner(REFINER, device)

## 1. Test set -> submission CSV

In [ ]:
paths = sorted(TEST_IMAGES.glob("*.jpeg"))
rows, seconds = [], []
for stem, masks, shape, dt in predict_all(paths, detector, refiner, cfg, device):
    rows += submission_rows(stem, masks, shape)
    seconds.append(dt)
out = ROOT / "outputs" / "submission.csv"
out.parent.mkdir(exist_ok=True)
out.write_text("filament_id,segmentation_rle\n" + "\n".join(rows) + "\n")
print(f"{out}: {len(rows)} rows, {len(paths)} images, {sum(seconds):.0f} s ({sum(seconds)/len(paths):.2f} s/image on {device})")

## 2. Validation split -> rubric metrics (PQ over every annotator record, IoU/Dice, 1:n and n:1)

In [ ]:
val = heldout_stems(ANN)
val_paths = [p for p in sorted(TRAIN_IMAGES.glob("*.jpeg")) if p.stem in val]
preds = {}
for stem, masks, shape, dt in predict_all(val_paths, detector, refiner, cfg, device):
    preds[stem] = [{"size": list(shape), "counts": encode(m).encode()} for m in masks]
ev = evaluate(load_records(ANN, val), preds)
ev.summary()